In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
sentence = ["I", "love", "deep", "learning"]
word_to_ix = {word: i for i, word in enumerate(sentence)}
ix_to_word = {i: word for word, i in word_to_ix.items()}


In [5]:
input_seq = [word_to_ix["I"], word_to_ix["love"], word_to_ix["deep"]]
target = word_to_ix["learning"]


In [6]:
input_tensor = torch.tensor(input_seq).unsqueeze(0) # shape: (1, 3)
target_tensor = torch.tensor([target]) # shape: (1)


In [7]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(RNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # use output from last timestep
        return out


In [8]:
vocab_size = len(word_to_ix)
model = RNNModel(vocab_size, embedding_dim=10, hidden_dim=20)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [9]:
for epoch in range(100):
  optimizer.zero_grad()
  output = model(input_tensor)
  loss = criterion(output, target_tensor)
  loss.backward()
  optimizer.step()
  if (epoch+1) % 10 == 0:
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 10, Loss: 0.0522
Epoch 20, Loss: 0.0061
Epoch 30, Loss: 0.0023
Epoch 40, Loss: 0.0014
Epoch 50, Loss: 0.0011
Epoch 60, Loss: 0.0010
Epoch 70, Loss: 0.0009
Epoch 80, Loss: 0.0008
Epoch 90, Loss: 0.0008
Epoch 100, Loss: 0.0007


In [10]:
with torch.no_grad():
  output = model(input_tensor)
  predicted_ix = torch.argmax(output, dim=1).item()
  print("Predicted word:", ix_to_word[predicted_ix])

Predicted word: learning
